In [1]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd()
current_dir

WindowsPath('E:/Openai/abtest/notebooks')

In [2]:
project_root = current_dir.parent
data_path = project_root / "data" / "raw" / "hillstrom.csv"

print(project_root)
print(data_path)
print(data_path.exists())

E:\Openai\abtest
E:\Openai\abtest\data\raw\hillstrom.csv
True


In [3]:
df = pd.read_csv(data_path)

print(type(df))
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
(64000, 12)


In [4]:
print(df.columns.tolist())
df.head()

['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  object 
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  object 
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  object 
 8   segment          64000 non-null  object 
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), object(4)
memory usage: 5.9+ MB


In [6]:
missing_counts = df.isna().sum()
missing_counts

recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64

In [7]:
df["segment"].value_counts(dropna=False)

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

In [8]:
df["visit"].value_counts(dropna=False)

visit
0    54606
1     9394
Name: count, dtype: int64

In [9]:
df["conversion"].value_counts(dropna=False)

conversion
0    63422
1      578
Name: count, dtype: int64

In [10]:
zero_spend_count = (df["spend"] == 0).sum()
negative_spend_count = (df["spend"] < 0).sum()

print("零消费人数：", zero_spend_count)
print("负金额记录数：", negative_spend_count)

零消费人数： 63422
负金额记录数： 0


In [11]:
purchase_mismatch = (df["conversion"] == 1) != (df["spend"] > 0)

print("购买标记与消费金额不一致的记录数：", purchase_mismatch.sum())

购买标记与消费金额不一致的记录数： 0


In [12]:
purchase_without_visit = (df["conversion"] == 1) & (df["visit"] == 0)

print("购买但未访问的记录数：", purchase_without_visit.sum())

购买但未访问的记录数： 0


In [13]:
df[["recency", "history", "spend"]].describe()

,recency,history,spend
count,64000.000000,64000.000000,64000.000000
mean,5.763734,242.085656,1.050908
std,3.507592,256.158608,15.036448
min,1.000000,29.990000,0.000000
25%,2.000000,64.660000,0.000000
50%,6.000000,158.110000,0.000000
75%,9.000000,325.657500,0.000000
max,12.000000,3345.930000,499.000000


In [14]:
zero_spend_rate = zero_spend_count / len(df)
print(f"零消费客户占比：{zero_spend_rate:.2%}")

零消费客户占比：99.10%


1. 数据有多少行、多少列，分为哪三组；
2. 零消费比例是多少，为什么零消费不等于缺失；
3. 两项逐行一致性检查发现了什么，以及这些检查尚不能证明什么。



1. 数据共有 64,000 行、12 列。按照 `segment` 分为 `Womens E-Mail`、`Mens E-Mail` 和 `No E-Mail` 三个实验组。
2. 零消费客户有 63,422 人，占全部客户的 99.10%。`spend=0` 是已记录的消费金额，表示该客户在观察期内没有消费，不属于缺失值；这些客户仍然要保留在总体人数和人均消费的分母中。
3. 购买标记与消费金额不一致的记录数为 0，购买但未访问的记录数也为 0，说明 `conversion`、`spend` 和 `visit` 在这两项规则下逐行一致。这些检查只能验证相关字段的内部对应关系，不能证明三组实验前客户特征已经平衡，也不能据此判断邮件是否产生了效果。